# Agents, Tools, MCPs and All That: Part 2
---------

## 1. Introduction
-------------------

## 2. CI/CD With Terraform & GitHub Actions
-------------------


## 3. Using A Remote MCP Server
-------------------

Tools are often tied to local code, which makes sharing hard. Additionally, each LLM and agent framework (OpenAI, Google, Anthropic, etc.) has their own way of defining and using tools. This can cause headaches for developers that need to make tools work across these various frameworks. The [**Model Context Protocol (MCP)**](https://en.wikipedia.org/wiki/Model_Context_Protocol) is an open standard for connecting LLM applications to external tools and data sources.

MCP provides a common interface by creating,

- **MCP servers**: Exposes tools, resources, or prompts.
- **MCP clients**: Connects an application or agent to the server.
- **Schemas**: Describes the tool's arguments and return values.

Model context protocol improves upon local tools by offering standard protocols for using tools on a remote server. Agents do not need direct access to data or source code the tool uses. In some ways, MCP is for agents as the way APIs are for regular code. The standard MCP framework is [FastMCP](https://gofastmcp.com/getting-started/welcome). I created a FastMCP server here with several tools I'll discuss. 

First, we'll create an MCP client and connect to our locally running server:

In [28]:
from fastmcp import Client
mcp_client = Client("https://agentsmcp-service-1025068371852.us-central1.run.app/mcp")

Then we can get a list of the tools,

In [20]:
async with mcp_client:
    tools = await mcp_client.list_tools()

tools

[Tool(name='get_weather', title='Get Weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', input_schema={'type': 'object', 'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city']}, execution=None, output_schema={'additionalProperties': True, 'type': 'object'}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}),
 Tool(name='convert_address_to_point', title='Convert Address To Point', description='Convert a street address to a Shapely Point object.', input_schema={'type': 'object', 'additionalProperties': False, 'properties': {'address': {'type':

In [21]:
tools[2].inputSchema

{'type': 'object',
 'additionalProperties': False,
 'properties': {'lat': {'type': 'number',
   'description': 'Latitude of the location.'},
  'lng': {'type': 'number', 'description': 'Longitude of the location.'}},
 'required': ['lat', 'lng']}

In [22]:
tools[2].outputSchema

{'properties': {'result': {'anyOf': [{'type': 'integer'}, {'type': 'null'}]}},
 'required': ['result'],
 'type': 'object',
 'x-fastmcp-wrap-result': True}

These tools are of type [FastMCP tools](https://gofastmcp.com/servers/tools),

In [23]:
type(tools[0])

mcp_types._types.Tool

We can then call a specific tool using the client's [call_tool](https://gofastmcp.com/clients/tools#execution-options) method

In [11]:
async with mcp_client:
    result = await mcp_client.call_tool("get_weather", {"city": "New York City"})

In [12]:
print(result)

CallToolResult(content=[TextContent(type='text', text='{"coord":{"lon":-74.006,"lat":40.7143},"weather":[{"id":800,"main":"Clear","description":"clear sky","icon":"01n"}],"base":"stations","main":{"temp":13.78,"feels_like":13.22,"temp_min":11.64,"temp_max":14.56,"pressure":1029,"humidity":77,"sea_level":1029,"grnd_level":1025},"visibility":10000,"wind":{"speed":2.24,"deg":113,"gust":4.02},"clouds":{"all":0},"dt":1793235404,"sys":{"type":2,"id":2111743,"country":"US","sunrise":1793186492,"sunset":1793224672},"timezone":-14400,"id":5128581,"name":"New York","cod":200}', annotations=None, meta=None)], structured_content={'coord': {'lon': -74.006, 'lat': 40.7143}, 'weather': [{'id': 800, 'main': 'Clear', 'description': 'clear sky', 'icon': '01n'}], 'base': 'stations', 'main': {'temp': 13.78, 'feels_like': 13.22, 'temp_min': 11.64, 'temp_max': 14.56, 'pressure': 1029, 'humidity': 77, 'sea_level': 1029, 'grnd_level': 1025}, 'visibility': 10000, 'wind': {'speed': 2.24, 'deg': 113, 'gust': 4.0

Notice that it returns a [CallToolResult](https://gofastmcp.com/clients/tools#structured-results). This is all pretty much the standard example, so I'll show a unique too, which is finding the closest public restroom to the point we received back:

In [29]:
async with mcp_client:
    data = await mcp_client.call_tool("convert_address_to_point", {"address": "265 Ocean Ave, Brooklyn, NY 11226"})

In [30]:
data.content

[TextContent(type='text', text='{"lat":40.6566465,"lng":-73.9617951}', annotations=None, meta=None)]

In [32]:
lat = data.data.get("lat")
lng = data.data.get("lng")

In [36]:
async with mcp_client:
    precinct_info = await mcp_client.call_tool("get_police_precinct", {"lat": lat, "lng": lng}) 

In [34]:
async with mcp_client:
    precinct_info = await mcp_client.call_tool("get_precinct_info", {"precinct_number": 17}) 

In [37]:
precinct_info.content

[TextContent(type='text', text='71', annotations=None, meta=None)]

In [61]:
restroom.data

{'facility_name': 'New Amsterdam Library, NYPL',
 'location_type': 'Library',
 'operator': 'NYPL',
 'status': 'Operational',
 'open': 'Year Round',
 'hours_of_operation': 'Sunday: Closed \nMonday: 10:00 am - 7:00 pm \nTuesday: 10:00 am - 7:00 pm  \nWednesday: 10:00 am - 7:00 pm   \nThursday: 10:00 am - 7:00 pm   \nFriday: 10:00 am - 5:00 pm \nSaturday: 10:00 am - 5:00 pm',
 'accessibility': 'Fully Accessible',
 'restroom_type': 'Single-Stall All Gender Restroom(s)',
 'changing_stations': 'Yes',
 'latitude': None,
 'longitude': None,
 'website': 'https://www.nypl.org/locations/new-amsterdam'}

Great! Next let's give a LangChain agent access to these tools!

## 5. Agents & MCPs
-------------------

Now in order to make an LangChain agent have access to the tools supplied by an MCP server we need to use a specialized MCP client for LangChain called [LangChain MCP Adapter](https://reference.langchain.com/python/langchain-mcp-adapters). This can be imported and set to the address of the MCP server and set transport type:

In [38]:
from langchain_mcp_adapters.client import MultiServerMCPClient  

lc_client = MultiServerMCPClient({"config": {"url": "https://agentsmcp-service-1025068371852.us-central1.run.app/mcp", "transport": "http"}})

ImportError: cannot import name 'RequestContext' from 'mcp.shared.context' (/Users/mikeharmon/Desktop/agentmcps/.venv/lib/python3.13/site-packages/mcp/shared/context.py)

Now we can get the tools:

In [66]:
tool_list = await lc_client.get_tools()
tool_list

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x12657fb00>),
 StructuredTool(name='convert_address_to_point', description='Convert a street address to a Shapely Point object.', args_schema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'descrip

Notice these are [LangChain tools](https://docs.langchain.com/oss/python/langchain/tools)! The adapter converted the FastMCP tool to a LangChain one:

In [67]:
type(tool_list[0])

langchain_core.tools.structured.StructuredTool

We can then use the tool as before,

In [72]:
temperature = await tool_list[0].ainvoke({"city":"Boston"})

The immediate return value isnt quite the same, its a list with json in it. We can convert back though,

In [77]:
import json
json.loads(temperature[0].get("text"))

{'coord': {'lon': -71.0598, 'lat': 42.3584},
 'weather': [{'id': 804,
   'main': 'Clouds',
   'description': 'overcast clouds',
   'icon': '04n'}],
 'base': 'stations',
 'main': {'temp': 22.4,
  'feels_like': 22.65,
  'temp_min': 20.64,
  'temp_max': 23.8,
  'pressure': 1017,
  'humidity': 75,
  'sea_level': 1017,
  'grnd_level': 1014},
 'visibility': 10000,
 'wind': {'speed': 2.57, 'deg': 140},
 'clouds': {'all': 86},
 'dt': 1787791539,
 'sys': {'type': 2,
  'id': 2007536,
  'country': 'US',
  'sunrise': 1787738592,
  'sunset': 1787786960},
 'timezone': -14400,
 'id': 4930956,
 'name': 'Boston',
 'cod': 200}

Now let's give the agent access to these tools from the MCP server. This just like giving the agent any other tools, 

In [78]:
agent = create_agent(model=model, tools=tool_list)

And now the agent can answer my question on the weather:

In [79]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'Whats the temparture in Boston?'
}]})
print(result.get("messages")[-1].content)

The current temperature in Boston is **about 22 °C** (≈ 72 °F).


Great!

Let's try something less straight forward. The [tool to find the police precinct](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py#L89) requires a latitude and longitude. If a users passes in an address the agent would need to conver the address to a latitude and longitude ([convert_address_to_point](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py#L57) tool) first and then pass that to the [get_police_precint tool](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py#L87). We can see this plainly with the entire message chain:

In [80]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What police precinct is 306 West 54th Street, manhattan, manhattan in?'
}]})

In [82]:
result.get("messages")

[HumanMessage(content='What police precinct is 306 West 54th Street, manhattan, manhattan in?', additional_kwargs={}, response_metadata={}, id='3fa87a68-b890-4b89-8a54-dfafc867ea54'),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "What police precinct is 306 West 54th Street, manhattan, manhattan in?" Need to determine precinct number. We have functions: convert_address_to_point, get_police_precinct. So first convert address to point, then get precinct. Use convert_address_to_point with address string. Then feed lat/lng to get_police_precinct. Then respond with precinct number and maybe info. Could also fetch precinct info using get_precinct_info for more detail. Let\'s do steps.\n\n', 'tool_calls': [{'id': 'fc_3d5d66e2-d976-410b-a5d2-ebc205032363', 'function': {'arguments': '{"address":"306 West 54th Street, Manhattan, NY"}', 'name': 'convert_address_to_point'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 143, 'prompt

We can see that agent reasoning:

In [87]:
print(result.get("messages")[1].additional_kwargs.get("reasoning_content"))

The user asks: "What police precinct is 306 West 54th Street, manhattan, manhattan in?" Need to determine precinct number. We have functions: convert_address_to_point, get_police_precinct. So first convert address to point, then get precinct. Use convert_address_to_point with address string. Then feed lat/lng to get_police_precinct. Then respond with precinct number and maybe info. Could also fetch precinct info using get_precinct_info for more detail. Let's do steps.




One thing to note is that the MCP tools take up space in the model context window which causes challenges. There are strategies to mitigate this, but I wont go over this here. The final message from the agent is,

In [90]:
print(result.get("messages")[-1].content)

The address **306 West 54th Street, Manhattan, NY** falls within **NYPD Precinct 18**.


Looking at the agent's actions in this way, a list of messages, is not ideal. This is where [LangSmith](https://www.langchain.com/langsmith-platform) comes into play.

<!--  -->

## 6. Agent Observability With Langsmith
-------------------

Parsing throught list of message is less than ideal. It also doesn't help us understand where and error occurred or what part of the process took the most amount of time. LangSmith offers as a way to do all of this through there interface. LangSmith integrates with most agent harnesses and integreates seamlessly with LangChain and LangGraph.

As an example I can force the LLM to take 3 tool calls to get me the phone number of the closest police precinct (this requires the precinct number be sent to [get_precinct_info](https://github.com/mdh266/langchain-fastmcp/blob/main/mymcp/server.py#L207C5-L207C22) tool). The query is:

In [91]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What is the phone number for closests police station to 306 West 54th Street, Manhattan?'
}]})

The agent traces on LangSmith show us the path,

<img src="https://github.com/mdh266/langchain-fastmcp/blob/main/images/trace1.png?raw=1">

The agent clearly goes from `query` -> `convert_address_to_point` -> `get_police_precinct` -> `get_precinct_info`.

Zooming into the last tool call, we can see that the agent figured out how to pass the police precinct number into the tool:

<img src="https://github.com/mdh266/langchain-fastmcp/blob/main/images/trace2.png?raw=1">

The agent receives the json result back from the tool and then generates a reponse that we see as,

In [92]:
print(result.get("messages")[-1].content)

The nearest NYPD precinct to 306 West 54th Street, Manhattan is the **Midtown North Precinct (Precinct 18)**.  

**Phone:** **212‑767‑8400**.


## 7. Next Steps
-------

In this post I went over how to create agents with Langchain and give them access to tools through FastMCP. Additionally we covered loggin the agent traces using LangSmith. This was all pretty simple proeject and the MCP Server is local. In a follow up post I'll show how to deploy this MCP server to Google Cloud and add it to Claude Code or another coding agent.  

Hope you enjoyed this!